In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import factorial

In [ ]:
WPIXSCALE = 0.059 # arcsec
ZPIXSCALE = 3. * WPIXSCALE
N, Nw, Nz = 120, 18, 12
R = np.sqrt((ZPIXSCALE * (3 + Nz / 2))**2 + (WPIXSCALE * (3 + Nw / 2))**2) + 0.03

xs = (WPIXSCALE / 2) * (np.arange(1 + np.int32(2 * R / (WPIXSCALE / 2))) - 0.5 * (1 + np.int32(2 * R / (WPIXSCALE / 2)) - 1.))
ygrid, xgrid = np.meshgrid(xs, xs)
rgrid_sc = np.sqrt(xgrid**2 + ygrid**2)
circle_mask = rgrid_sc > R
phigrid_sc = np.atan2(ygrid, xgrid)

def make_circle(arr):
    arr[circle_mask] = np.nan
    return arr

In [ ]:
def explicit_radial(n, m, r):
    m = np.abs(m)
    if ((n - m) % 2 == 0):
        k = np.arange(np.int32((n - m) / 2) + 1)
        big = r**(n - 2 * k) * ((-1)**k * factorial(n - k)) / (factorial(k) * factorial(((n + m) / 2) - k) * factorial(((n - m) / 2) - k))
        return np.sum(big)
    else:
        return 0

def angular(m, theta):
    return np.exp(1j * m * theta)

def explicit_zernike(n, m, r, phi):
    temp = np.zeros(r.shape)
    for i in range(r.shape[0]):
        for j in range(r.shape[1]):
            temp[i,j] = shitR(n, m, r[i,j])
    return temp * angular(m, phi)

In [ ]:
rs = rgrid_sc / R
phis = phigrid_sc

In [ ]:
def recur(n, m, r, r4, r2):
    return (2 * (n - 1) * (2 * n * (n - 2) * r**2 - m**2 - n * (n - 2)) * r2 - n * (n + m - 2) * (n - m - 2) * r4) \
    / ((n + m) * (n - m) * (n - 2))

def Rmm(m, r):
    return r**m

def Rmm2(m, r):
    return ((m + 2) * r**2 - (m + 1)) * r**m

In [ ]:
"""
R00 = Rmm(0, rs)
R20 = Rmm2(0, rs)
R40 = recur(4, 0, rs, R00, R20)
R60 = recur(6, 0, rs, R20, R40)
R80 = recur(8, 0, rs, R40, R60)
R100 = recur(10, 0, rs, R60, R80)
"""

In [ ]:
"""
plt.imshow(make_circle(R00), cmap="RdBu")
plt.show()
plt.imshow(make_circle(R20), cmap="RdBu")
plt.show()
plt.imshow(make_circle(R40), cmap="RdBu")
plt.show()
plt.imshow(make_circle(R60), cmap="RdBu")
plt.show()
plt.imshow(make_circle(R80), cmap="RdBu")
plt.show()
plt.imshow(make_circle(R100), cmap="RdBu")
plt.show()
"""

In [ ]:
"""
R11 = Rmm(1, rs)
R31 = Rmm2(1, rs)
R51 = recur(5, 1, rs, R11, R31)
R71 = recur(7, 1, rs, R31, R51)
R91 = recur(9, 1, rs, R51, R71)
R111 = recur(11, 1, rs, R71, R91)
"""

In [ ]:
"""
plt.imshow(make_circle(R11 * angular(1, phis)).real, cmap="RdBu")
plt.show()
plt.imshow(make_circle(R31 * angular(1, phis)).real, cmap="RdBu")
plt.show()
plt.imshow(make_circle(R51 * angular(1, phis)).real, cmap="RdBu")
plt.show()
plt.imshow(make_circle(R71 * angular(1, phis)).real, cmap="RdBu")
plt.show()
plt.imshow(make_circle(R91 * angular(1, phis)).real, cmap="RdBu")
plt.show()
plt.imshow(make_circle(R111 * angular(1, phis)).real, cmap="RdBu")
plt.show()
"""

In [ ]:
Nn = 10
zernikes = np.full((Nn, Nn, rs.shape[0], rs.shape[1]), np.nan)
for i in range(Nn):
    zernikes[i,i] = Rmm(i, rs)
    if i + 2 < Nn:
        zernikes[i + 2,i] = Rmm2(i, rs)
    for j in range(i):
        if (i - 4 >= j and (i - j) % 2 == 0):
            zernikes[i, j] = recur(i, j, rs, zernikes[i - 4, j], zernikes[i - 2, j])

fig, ax = plt.subplots(10, 10, figsize=(10, 10))
for i in range(Nn):
    for j in range(Nn):
        ax[i,j].imshow(make_circle(zernikes[i,j] * angular(j, phis)).real, origin="lower", interpolation="nearest", cmap="RdBu")
        ax[i,j].axis("off")
plt.tight_layout()
plt.show()

"""
fig, ax = plt.subplots(Nn, np.int32(Nn / 2), figsize=(8, 10))
for i in range(Nn):
    for j in range(np.int32(Nn / 2)):
        ax[i,j].imshow(make_circle(zernikes[i, (i % 2) + (2 * j)] * angular((i % 2) + (2 * j), phis)).real, origin="lower", interpolation="nearest", cmap="RdBu")
        ax[i,j].axis("off")
plt.tight_layout()
plt.show()
"""